# Day 2: Feature Engineering Pipeline

**Training exercise: Engineer 6+ features with recency, frequency, and interaction types**

Run the notebook from top to bottom to generate customer features, metadata, and the final CSV output.


## Environment Setup

Import the libraries used throughout the feature engineering exercise.


In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
print("Loading libraries...")


print("✓ Libraries loaded: pandas, numpy, json, datetime")

Loading libraries...
✓ Libraries loaded: pandas, numpy, json, datetime


## Part 1: Setup - Create Customer Support Dataset


In [2]:
# ============================================================================
# PART 1: SETUP - Create customer support dataset
# ============================================================================

print("="*80)
print("DAY 2: FEATURE ENGINEERING PIPELINE")
print("="*80)

np.random.seed(42)

# Generate synthetic customer support data
n_customers = 100
base_date = datetime(2024, 1, 1)

data = {
    'customer_id': range(1001, 1001 + n_customers),
    'first_contact_date': [base_date - timedelta(days=np.random.randint(30, 730)) for _ in range(n_customers)],
    'last_contact_date': [base_date - timedelta(days=np.random.randint(0, 60)) for _ in range(n_customers)],
    'last_purchase_date': [base_date - timedelta(days=np.random.randint(0, 180)) for _ in range(n_customers)],
    'total_support_tickets': np.random.randint(0, 50, n_customers),
    'total_purchases': np.random.randint(0, 20, n_customers),
    'purchase_amount': np.random.uniform(100, 10000, n_customers),
    'churned': np.random.binomial(1, 0.25, n_customers)  # 25% churn rate
}

df_raw = pd.DataFrame(data)

print("\nDataset created:")
print(df_raw.head())
print(f"\nDataset shape: {df_raw.shape}")
print(f"Churned rate: {df_raw['churned'].mean():.1%}")



DAY 2: FEATURE ENGINEERING PIPELINE

Dataset created:
   customer_id first_contact_date last_contact_date last_purchase_date  \
0         1001         2023-08-22        2023-12-16         2023-07-31   
1         1002         2022-09-23        2023-12-25         2023-08-18   
2         1003         2023-03-07        2023-11-28         2023-11-01   
3         1004         2023-08-18        2023-11-28         2023-07-21   
4         1005         2023-09-22        2023-11-30         2023-11-12   

   total_support_tickets  total_purchases  purchase_amount  churned  
0                     39                9      4636.703616        0  
1                     19                2      5501.606214        0  
2                     34               17      9420.501607        1  
3                     47               12      3922.416114        0  
4                     24                6      9615.786582        0  

Dataset shape: (100, 8)
Churned rate: 35.0%


## Part 2: Live Demo - Example Features (2-3 Features Shown)


In [3]:
# ============================================================================
# PART 2: LIVE DEMO - Example Features (2-3 features shown)
# ============================================================================

print("\n" + "="*80)
print("LIVE DEMO: Example Feature Engineering")
print("="*80)

df_features = df_raw.copy()

# Feature 1: Recency - Days since last contact
print("\nFeature 1: days_since_last_contact")
df_features['days_since_last_contact'] = (base_date - df_features['last_contact_date']).dt.days
print(f"  Example values: {df_features['days_since_last_contact'].head().tolist()}")
print(f"  Why? Recent contact = engaged customer = less likely to churn")

# Feature 2: Frequency - Support tickets per month
print("\nFeature 2: support_tickets_per_month")
df_features['months_active'] = ((base_date - df_features['first_contact_date']).dt.days / 30).clip(lower=1)
df_features['support_tickets_per_month'] = df_features['total_support_tickets'] / df_features['months_active']
print(f"  Example values: {df_features['support_tickets_per_month'].head().tolist()}")
print(f"  Why? High support tickets + low purchases = unhappy customer")




LIVE DEMO: Example Feature Engineering

Feature 1: days_since_last_contact
  Example values: [16, 7, 34, 34, 32]
  Why? Recent contact = engaged customer = less likely to churn

Feature 2: support_tickets_per_month
  Example values: [8.863636363636363, 1.2258064516129032, 3.4, 10.367647058823529, 7.128712871287129]
  Why? High support tickets + low purchases = unhappy customer


## Part 3: Independent Exercise - Recency Features


In [4]:
# ============================================================================
# PART 3: INDEPENDENT EXERCISE - Recency Features
# ============================================================================

print("\n" + "="*80)
print("EXERCISE: Task 1 - Create Recency Features")
print("="*80)
print("TODO: Create the following recency features:")
print("  1. days_since_first_contact = days between first contact and today")
print("  2. days_since_last_purchase = days between last purchase and today")
print("\nHint: Use (base_date - df_features['column_name']).dt.days")

# STUDENT CODE HERE:
df_features['days_since_first_contact'] = (base_date - df_features['first_contact_date']).dt.days
df_features['days_since_last_purchase'] = (base_date - df_features['last_purchase_date']).dt.days

print(f"\n✓ Recency features created:")
print(f"  days_since_first_contact: {df_features['days_since_first_contact'].describe()}")
print(f"  days_since_last_purchase: {df_features['days_since_last_purchase'].describe()}")




EXERCISE: Task 1 - Create Recency Features
TODO: Create the following recency features:
  1. days_since_first_contact = days between first contact and today
  2. days_since_last_purchase = days between last purchase and today

Hint: Use (base_date - df_features['column_name']).dt.days

✓ Recency features created:
  days_since_first_contact: count    100.000000
mean     352.730000
std      195.801428
min       31.000000
25%      181.500000
50%      363.500000
75%      505.250000
max      729.000000
Name: days_since_first_contact, dtype: float64
  days_since_last_purchase: count    100.000000
mean     105.080000
std       48.530253
min        1.000000
25%       65.500000
50%      113.500000
75%      146.000000
max      179.000000
Name: days_since_last_purchase, dtype: float64


## Part 4: Independent Exercise - Frequency Features


In [5]:
# ============================================================================
# PART 4: INDEPENDENT EXERCISE - Frequency Features
# ============================================================================

print("\n" + "="*80)
print("EXERCISE: Task 2 - Create Frequency Features")
print("="*80)
print("TODO: Create the following frequency features:")
print("  1. purchases_per_month = total purchases / months active")
print("  2. contact_frequency = total support tickets / months active")
print("\nHint: Use the 'months_active' column already created")

# Also CODE HERE:
df_features['purchases_per_month'] = df_features['total_purchases'] / df_features['months_active']
df_features['contact_frequency'] = df_features['total_support_tickets'] / df_features['months_active']

print(f"\n✓ Frequency features created:")
print(f"  purchases_per_month: {df_features['purchases_per_month'].describe()}")
print(f"  contact_frequency: {df_features['contact_frequency'].describe()}")




EXERCISE: Task 2 - Create Frequency Features
TODO: Create the following frequency features:
  1. purchases_per_month = total purchases / months active
  2. contact_frequency = total support tickets / months active

Hint: Use the 'months_active' column already created

✓ Frequency features created:
  purchases_per_month: count    100.000000
mean       1.403105
std        1.831783
min        0.000000
25%        0.356241
50%        0.851303
75%        1.703571
max       10.800000
Name: purchases_per_month, dtype: float64
  contact_frequency: count    100.000000
mean       4.634815
std        6.610814
min        0.061475
25%        1.260776
50%        2.100256
75%        4.496700
max       33.488372
Name: contact_frequency, dtype: float64


## Part 5: Independent Exercise - Interaction Features


In [6]:
# ============================================================================
# PART 5: INDEPENDENT EXERCISE - Interaction Features
# ============================================================================

print("\n" + "="*80)
print("EXERCISE: Task 3 - Create Interaction Features")
print("="*80)
print("TODO: Create the following interaction features:")
print("  1. support_intensity = total_support_tickets / (total_purchases + 1)")
print("     (High support + low purchases = unhappy customer)")
print("  2. support_to_spending_ratio = support_tickets_per_month / (purchases_per_month + 0.1)")
print("     (Are they buying as much given the support needed?)")
print("\nHint: Be careful with division by zero!")

# STUDENT CODE HERE:
df_features['support_intensity'] = df_features['total_support_tickets'] / (df_features['total_purchases'] + 1)
df_features['support_to_spending_ratio'] = (
    df_features['support_tickets_per_month'] / (df_features['purchases_per_month'] + 0.1)
)

print(f"\n✓ Interaction features created:")
print(f"  support_intensity: {df_features['support_intensity'].describe()}")
print(f"  support_to_spending_ratio: {df_features['support_to_spending_ratio'].describe()}")




EXERCISE: Task 3 - Create Interaction Features
TODO: Create the following interaction features:
  1. support_intensity = total_support_tickets / (total_purchases + 1)
     (High support + low purchases = unhappy customer)
  2. support_to_spending_ratio = support_tickets_per_month / (purchases_per_month + 0.1)
     (Are they buying as much given the support needed?)

Hint: Be careful with division by zero!

✓ Interaction features created:
  support_intensity: count    100.000000
mean       5.249142
std        7.255089
min        0.062500
25%        1.500000
50%        2.696691
75%        4.930556
max       38.000000
Name: support_intensity, dtype: float64
  support_to_spending_ratio: count    100.000000
mean       5.302458
std        8.929517
min        0.060144
25%        1.478170
50%        2.551655
75%        4.852272
max       69.421488
Name: support_to_spending_ratio, dtype: float64


## Part 6: Independent Exercise - Derived Features


In [7]:
# ============================================================================
# PART 6: INDEPENDENT EXERCISE - Derived Features
# ============================================================================

print("\n" + "="*80)
print("EXERCISE: Task 4 - Create Derived/Domain Features")
print("="*80)
print("TODO: Create business-logic features:")
print("  1. at_risk_flag = (days_since_last_contact > 30) & (support_tickets_per_month > 2)")
print("     (Haven't contacted in 30 days + complaining frequently = at risk)")
print("  2. engagement_score = 100 - (days_since_last_contact / 7)")
print("     (Higher = more recently engaged)")

# Also CODE HERE:
df_features['at_risk_flag'] = (
    (df_features['days_since_last_contact'] > 30) & 
    (df_features['support_tickets_per_month'] > 2)
).astype(int)

df_features['engagement_score'] = (100 - (df_features['days_since_last_contact'] / 7)).clip(lower=0)

print(f"\n✓ Derived features created:")
print(f"  at_risk_flag: {df_features['at_risk_flag'].sum()} customers at risk")
print(f"  engagement_score: {df_features['engagement_score'].describe()}")




EXERCISE: Task 4 - Create Derived/Domain Features
TODO: Create business-logic features:
  1. at_risk_flag = (days_since_last_contact > 30) & (support_tickets_per_month > 2)
     (Haven't contacted in 30 days + complaining frequently = at risk)
  2. engagement_score = 100 - (days_since_last_contact / 7)
     (Higher = more recently engaged)

✓ Derived features created:
  at_risk_flag: 30 customers at risk
  engagement_score: count    100.000000
mean      95.738571
std        2.397179
min       91.571429
25%       94.107143
50%       95.500000
75%       98.000000
max      100.000000
Name: engagement_score, dtype: float64


## Part 7: Feature Versioning & Metadata


In [8]:
# ============================================================================
# PART 7: FEATURE VERSIONING & METADATA
# ============================================================================

print("\n" + "="*80)
print("EXERCISE: Task 5 - Create Feature Metadata & Versioning")
print("="*80)
print("TODO: Document all engineered features in a metadata dict")

# List of engineered features
engineered_features = [
    'days_since_first_contact',
    'days_since_last_contact',
    'days_since_last_purchase',
    'support_tickets_per_month',
    'purchases_per_month',
    'contact_frequency',
    'support_intensity',
    'support_to_spending_ratio',
    'at_risk_flag',
    'engagement_score'
]

# Also CODE HERE:
feature_metadata = {
    "version": "1.0",
    "created_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "description": "Customer churn prediction features",
    "total_features": len(engineered_features),
    "features": [
        {
            "name": "days_since_first_contact",
            "type": "recency",
            "description": "Days between customer's first contact and today",
            "interpretation": "Higher = older customer"
        },
        {
            "name": "days_since_last_contact",
            "type": "recency",
            "description": "Days since last support contact",
            "interpretation": "Higher = disengaged"
        },
        {
            "name": "days_since_last_purchase",
            "type": "recency",
            "description": "Days since last purchase",
            "interpretation": "Higher = inactive buyer"
        },
        {
            "name": "support_tickets_per_month",
            "type": "frequency",
            "description": "Average support tickets filed per month",
            "interpretation": "Higher = more problems"
        },
        {
            "name": "purchases_per_month",
            "type": "frequency",
            "description": "Average purchases per month",
            "interpretation": "Higher = active buyer"
        },
        {
            "name": "contact_frequency",
            "type": "frequency",
            "description": "Total contacts divided by months active",
            "interpretation": "Normalized engagement metric"
        },
        {
            "name": "support_intensity",
            "type": "interaction",
            "description": "Support tickets per purchase",
            "interpretation": "High = problematic purchases"
        },
        {
            "name": "support_to_spending_ratio",
            "type": "interaction",
            "description": "Support frequency relative to purchase frequency",
            "interpretation": "High = lots of support for few purchases"
        },
        {
            "name": "at_risk_flag",
            "type": "derived",
            "description": "Binary flag: inactive + complaining = at risk",
            "interpretation": "1 = at risk of churn"
        },
        {
            "name": "engagement_score",
            "type": "derived",
            "description": "Normalized engagement (0-100 scale)",
            "interpretation": "Higher = more engaged"
        }
    ]
}

# Save metadata to JSON
with open('feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"\n✓ Feature metadata saved to feature_metadata.json")
print(f"✓ Total engineered features: {feature_metadata['total_features']}")
print(f"✓ Feature types: recency, frequency, interaction, derived")




EXERCISE: Task 5 - Create Feature Metadata & Versioning
TODO: Document all engineered features in a metadata dict

✓ Feature metadata saved to feature_metadata.json
✓ Total engineered features: 10
✓ Feature types: recency, frequency, interaction, derived


## Part 8: Feature Analysis & Correlation with Target


In [9]:
# ============================================================================
# PART 8: FEATURE ANALYSIS & CORRELATION WITH TARGET
# ============================================================================

print("\n" + "="*80)
print("FEATURE ANALYSIS: Correlation with Churn Target")
print("="*80)

# Calculate correlation with churn
correlations = df_features[engineered_features + ['churned']].corr()['churned'].drop('churned').sort_values()

print("\nFeature Correlations with Churn (negative = protects against churn):")
for feat, corr in correlations.items():
    print(f"  {feat:30s}: {corr:+.3f}")

print("\nInterpretation:")
print("  Negative correlation = feature reduces churn risk (good!)")
print("  Positive correlation = feature increases churn risk (warning!)")




FEATURE ANALYSIS: Correlation with Churn Target

Feature Correlations with Churn (negative = protects against churn):
  days_since_last_purchase      : -0.061
  support_tickets_per_month     : -0.030
  contact_frequency             : -0.030
  engagement_score              : -0.015
  support_to_spending_ratio     : -0.014
  purchases_per_month           : +0.003
  days_since_last_contact       : +0.015
  at_risk_flag                  : +0.023
  days_since_first_contact      : +0.064
  support_intensity             : +0.072

Interpretation:
  Negative correlation = feature reduces churn risk (good!)
  Positive correlation = feature increases churn risk (warning!)


## Part 9: Checkpoint - Final Output


In [10]:
# ============================================================================
# PART 9: CHECKPOINT - FINAL OUTPUT
# ============================================================================

print("\n" + "="*80)
print("CHECKPOINT: Verify Your Work")
print("="*80)

# Select only engineered features
df_final = df_features[['customer_id', 'churned'] + engineered_features]

print(f"\n✓ Final dataset shape: {df_final.shape}")
print(f"✓ Number of engineered features: {len(engineered_features)}")
print(f"\nFirst 5 rows:")
print(df_final.head())

# Save to CSV
try:
    df_final.to_csv('customer_features.csv', index=False)
except Exception as e:
    print(f"Error saving file: {e}")
print(f"\n✓ Features saved to customer_features.csv")

print("\n" + "="*80)
print("DAY 2 EXERCISE COMPLETE!")
print("="*80)
print("\nLearning Takeaways:")
print("  1. Recency, Frequency, Interaction = core feature types")
print("  2. Domain knowledge matters: at_risk_flag came from business insights")
print("  3. Always document features (metadata helps others understand)")
print("  4. Check correlations to validate features make sense")
print("\nNext: Tomorrow we'll use these features in a time series forecasting model!")




CHECKPOINT: Verify Your Work

✓ Final dataset shape: (100, 12)
✓ Number of engineered features: 10

First 5 rows:
   customer_id  churned  days_since_first_contact  days_since_last_contact  \
0         1001        0                       132                       16   
1         1002        0                       465                        7   
2         1003        1                       300                       34   
3         1004        0                       136                       34   
4         1005        0                       101                       32   

   days_since_last_purchase  support_tickets_per_month  purchases_per_month  \
0                       154                   8.863636             2.045455   
1                       136                   1.225806             0.129032   
2                        61                   3.400000             1.700000   
3                       164                  10.367647             2.647059   
4                    